In [1]:
%run MO_Atm_coupling_101325_test.ipynb


In [ ]:
#CHILI data from PROTEUS T1b run @ 1e5 yrs, tau=5
P_s_bar = 295.00136099
P_s = P_s_bar*1e5
T_s_prt = 1874.0125441
T_eq = 716.7151921485
T_int = 382.4810857560
M_test = 1.
R_test = 1.


In [ ]:
#these are used to convert to relative atomic abundances in the atmosphere.
#not the absolute masses.
M_C_atm = 4.0936538061e20
M_H_atm = 5.8153258795e17
H_O_atm = 1.0672269605e21
m_CNHO_atm = np.array([M_C_atm, 0., M_H_atm, H_O_atm])

In [ ]:
#running the iterator to find solution for MO + Atmosphere, converging to 
#the MO surface mass & radius with guesses of total mass and radius at 
#the top of the atmosphere.
out = calc_rad_atm_MO_spec_CNH_Teq_fixed_Atm_CNH(
            M_test, R_test, 
            T_eq, T_int,
            P_s,
            m_CNHO_atm, 
            initial_guess = [1.0004, 1.07]
            )


Iter  22
MR_top	[1.0002628 1.0572586]
rel_err	[6.45891562e-10 5.09201460e-07]
M_s 1.0
M_toa 1.0002627985974706
R_toa 1.0572585978554114
yPs_true [9.98894490e+02 2.65866117e+03 2.76928624e+07 1.55696608e+06
 1.64722863e-07 2.46650087e+05 0.00000000e+00]
M_s_true, R_s_true, CMF 1.0000000004243164 1.0000003135683355 0.24590622816496208
M_MO_frac 0.22357621882325493
Relative error: 
 [4.24316360e-10 3.13568336e-07]


In [ ]:
#calculate the MO volume fraction from the output object
def find_MO_sol_index(out_obj):
    delta_M = (out_obj.M_s - out_obj.mantle_profiles['M']/M_E)- \
            out_obj.M_MO_frac * out_obj.M_s
    temp1 = delta_M[1:]*delta_M[:-1]
    sol_index = np.where(temp1 < 0)[0][0]
    return sol_index, delta_M

def calc_V_frac_MO(out):
    sol_index, delta_M = find_MO_sol_index(out)
    factor = delta_M[sol_index+1]/(delta_M[sol_index+1]-delta_M[sol_index])
    r_MO = out.mantle_profiles['r'][sol_index + 1] + \
        factor*(out.mantle_profiles['r'][sol_index] - out.mantle_profiles['r'][sol_index + 1])
    r_MO_E = r_MO / R_E
    V_frac_MO = 1. - (r_MO_E/out.R_s_true)**3.
    return V_frac_MO

In [6]:
V_frac_MO = calc_V_frac_MO(out)
print('MO volume fraction:', V_frac_MO)

MO volume fraction: 0.29281250485853605


In [ ]:
#save CHILI-style output to CSV given MOAChi output object
def save_CHILI_output_CSV(out, filename):
    #y_O2, y_H2, y_CO2, y_CO, y_CH4, y_H2O
    zs = (out.rs - out.R_s_true)[1:] * R_E
    Ps = out.PTMtaus_a[0][1:]/1e5
    Ts = out.PTMtaus_a[1][1:]
    yss = out.LO[0]
    Pss = Ps[np.newaxis, :] * yss
    #append surface values
    zs[-1] = 0.
    Ps[-1] = out.PTMtau_s[0]/1e5
    Ts[-1] = out.PTMtau_s[1]
    Pss[:,-1] = out.yPs_surf/1e5
    
    #reorder Pss gas species into H2O, CO2, CO, H2, CH4, O2
    Pss_reordered = np.zeros_like(Pss[1:])

    Pss_reordered[0,:] = Pss[5,:]  #H2O
    Pss_reordered[1,:] = Pss[2,:]  #CO2
    Pss_reordered[2,:] = Pss[3,:]  #CO
    Pss_reordered[3,:] = Pss[1,:]  #H2
    Pss_reordered[4,:] = Pss[4,:]  #CH4
    Pss_reordered[5,:] = Pss[0,:]  #O2
    data_to_save = np.concatenate((zs[np.newaxis, :], Ps[np.newaxis, :],
                                    Ts[np.newaxis, :], Pss_reordered), axis=0)
    header_str = 'z(m),p_tot(bar),T(K),p_H2O(bar),p_CO2(bar),p_CO(bar),p_H2(bar),p_CH4(bar),p_O2(bar)'
    np.savetxt(filename, data_to_save.T, delimiter=',', header=header_str)
    print(f'Saved CHILI-style output to {filename}')


In [22]:
save_CHILI_output_CSV(out, 'static-moachi-trappist1b-tau5-proteus-data.csv')

Saved CHILI-style output to static-moachi-trappist1b-tau5-proteus-data.csv
